# 01 — Ingesta Bronze

Propósito: cargar todas las fuentes de datos crudas, verificar que se
leen correctamente, documentar la cobertura temporal y guardar snapshots
en Parquet. Este notebook no limpia ni transforma datos.

## IMPORTS Y CONFIGURACIÓN DE RUTAS

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


from src.tfm_io import (
    read_article_sales,
    read_articles,
    read_menu,
    read_departments,
    read_tickets,
    read_tips,
    read_reservations,
    read_data_file,
    load_data_directory,
    summarize_datasets,
)


DATA_DIR = PROJECT_ROOT / 'data'
BRONZE     = DATA_DIR / 'bronze'
VENTAS     = BRONZE / 'ventas_semanal'
TICKETS    = BRONZE / 'pdfs'
SILVER     = DATA_DIR / 'silver'


print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')

Project root: /Users/laura/TFM-Hosteleria-AI
Data directory: /Users/laura/TFM-Hosteleria-AI/data


## 1. Carga de fuentes transaccionales

In [2]:
# cargamos la lista de tickets utilizando la función read_tickets -> devuelve un dataframe
df_tickets = read_tickets(BRONZE / 'LISTA_TICKETS.xls') 

# comprobamos que se han cargado bien
display(df_tickets.head(10))
display(df_tickets.dtypes)
display(df_tickets.shape)

,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,date,document_id,document_total,receipt_count
0,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000001,7.00,1
1,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000002,65.40,1
2,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000003,8.20,1
3,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000004,77.10,1
4,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000005,35.05,1
5,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000006,83.50,1
6,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000007,85.25,1
7,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000008,96.60,1
8,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000009,16.40,1
9,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000010,24.00,1


source_file                       str
report_start           datetime64[us]
report_end             datetime64[us]
report_generated_on    datetime64[us]
terminal_start                    str
terminal_end                      str
turn                              str
date                   datetime64[us]
document_id                    string
document_total                float64
receipt_count                   Int64
dtype: object

(6709, 11)

In [3]:
# cargamos las ventas semanales utilizando load_data_directory, porque todos los archivos de la carpeta son del mismo tipo
# La función devuelve:

# 1. Un diccionario de DataFrames.
# 2. Un DataFrame con los errores de carga detectados.

datasets, load_errors = load_data_directory(
    VENTAS
)

if not load_errors.empty:
    print('Files with loading errors:')

    display(load_errors)

else:
    print('All files were loaded correctly.')

summary = summarize_datasets(
    datasets
)

display(summary)

All files were loaded correctly.


,dataset,rows,columns
0,article_sales_periodic,5532,13


In [4]:
# comprobamos que se han cargado todos y que no tenemos huecos en los datos -> están todas las semanas

df_ventas = datasets["article_sales_periodic"]

# Extraer una fila por fichero con su rango de fechas
semanas = (
    df_ventas
    .groupby("source_file")
    .agg(
        inicio=("report_start", "first"),
        fin=("report_end", "first"),
    )
    .reset_index()
    .sort_values("inicio")
    .reset_index(drop=True)
)

# Comparar cada semana con la anterior
huecos = []
for i in range(1, len(semanas)):
    fin_anterior   = semanas.loc[i - 1, "fin"]
    inicio_actual  = semanas.loc[i, "inicio"]
    dias_de_hueco  = (inicio_actual - fin_anterior).days - 1

    if dias_de_hueco > 0:
        huecos.append({
            "entre":      semanas.loc[i - 1, "source_file"],
            "y":          semanas.loc[i, "source_file"],
            "desde":      fin_anterior + pd.Timedelta(days=1),
            "hasta":      inicio_actual - pd.Timedelta(days=1),
            "dias_falta": dias_de_hueco,
        })

if huecos:
    print(f"⚠️  {len(huecos)} hueco(s) detectado(s) en ventas semanales:")
    display(pd.DataFrame(huecos))
else:
    print("✓ Sin huecos entre semanas.")

✓ Sin huecos entre semanas.


In [5]:
# cargamos las reservas utilizando la función read_reservations -> devuelve un dataframe

df_reservas = read_reservations(BRONZE / 'RESERVAS.xlsx') 

# comprobamos que se han cargado bien
display(df_reservas.head(10))
display(df_reservas.dtypes)
display(df_reservas.shape)

,source_file,reservation_datetime,created_datetime,reservation_date,reservation_time,status,shift,people,origin,referrer,created_date,created_time,restaurant,reservation_type,table,zone,entered_by,group,reference,reference_code
0,RESERVAS.xlsx,2022-11-10 21:00:00,2022-11-10 17:41:00,2022-11-10,21:00:00,No show,Cena,2,software,NaN,2022-11-10,17:41:00,LA ROCA Pozuelo,Normal,202,Terraza Cubierta,NaN,NO,NaN,EI48iz
1,RESERVAS.xlsx,2022-11-10 22:00:00,2022-11-10 17:48:36,2022-11-10,22:00:00,Cancelado por el cliente,Cena,4,software,NaN,2022-11-10,17:48:36,LA ROCA Pozuelo,Normal,4,Sala,NaN,NO,NaN,piBz3M
2,RESERVAS.xlsx,2022-11-10 20:30:00,2022-11-10 18:13:25,2022-11-10,20:30:00,Liberada,Cena,4,software,NaN,2022-11-10,18:13:25,LA ROCA Pozuelo,Normal,4,Sala,NaN,NO,NaN,5bgY71
3,RESERVAS.xlsx,2022-11-10 20:30:00,2022-11-10 18:14:01,2022-11-10,20:30:00,Cancelado por el cliente,Cena,4,software,NaN,2022-11-10,18:14:01,LA ROCA Pozuelo,Normal,10,Sala,NaN,NO,NaN,3M42ao
4,RESERVAS.xlsx,2022-11-13 13:30:00,2022-11-13 13:53:13,2022-11-13,13:30:00,Liberada,Comida,8,software,NaN,2022-11-13,13:53:13,LA ROCA Pozuelo,Normal,"14, 15",Sala,NaN,NO,NaN,mhxgix
5,RESERVAS.xlsx,2022-11-15 21:00:00,2022-11-15 16:31:47,2022-11-15,21:00:00,Liberada,Cena,2,software,NaN,2022-11-15,16:31:47,LA ROCA Pozuelo,Normal,5,Sala,NaN,NO,NaN,I8ZyTs
6,RESERVAS.xlsx,2022-11-16 20:45:00,2022-11-15 21:21:22,2022-11-16,20:45:00,Liberada,Cena,6,software,NaN,2022-11-15,21:21:22,LA ROCA Pozuelo,Normal,"14, 15",Sala,NaN,NO,NaN,wXm6aG
7,RESERVAS.xlsx,2022-11-18 21:15:00,2022-11-15 22:08:49,2022-11-18,21:15:00,No show,Cena,7,software,NaN,2022-11-15,22:08:49,LA ROCA Pozuelo,Normal,2026-06-07 00:00:00,Sala,NaN,NO,NaN,xZAUjI
8,RESERVAS.xlsx,2022-11-18 21:15:00,2022-11-15 22:12:31,2022-11-18,21:15:00,Liberada,Cena,4,software,NaN,2022-11-15,22:12:31,LA ROCA Pozuelo,Normal,4,Sala,NaN,NO,NaN,wzpR3n
9,RESERVAS.xlsx,2022-11-20 14:00:00,2022-11-15 22:24:10,2022-11-20,14:00:00,Cancelado por el cliente,Comida,7,software,NaN,2022-11-15,22:24:10,LA ROCA Pozuelo,Normal,1,Sala,NaN,NO,NaN,jfCchi


source_file                        str
reservation_datetime    datetime64[us]
created_datetime        datetime64[us]
reservation_date        datetime64[us]
reservation_time                object
status                          object
shift                           object
people                           Int64
origin                          object
referrer                        object
created_date            datetime64[us]
created_time                    object
restaurant                      object
reservation_type                object
table                           string
zone                            object
entered_by                      object
group                           object
reference                       object
reference_code                  object
dtype: object

(21341, 20)

In [6]:
# cargamos las propinas utilizando la función read_tips -> devuelve un dataframe

df_tips = read_tips(BRONZE / 'PROPINAS_TICKET.xls') 

# comprobamos que se han cargado bien
display(df_tips.head(10))
display(df_tips.dtypes)
display(df_tips.shape)

,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,document_id,document_amount,tip,document_total
0,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000015,65.45,0.55,66.0
1,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000020,93.45,3.55,97.0
2,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000022,124.90,1.10,126.0
3,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000035,49.05,0.95,50.0
4,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000036,76.30,3.70,80.0
5,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000039,38.40,1.10,39.5
6,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000041,245.30,6.70,252.0
7,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000047,117.70,7.30,125.0
8,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000048,99.70,4.30,104.0
9,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00000049,68.70,3.30,72.0


source_file                       str
report_start           datetime64[us]
report_end             datetime64[us]
report_generated_on    datetime64[us]
terminal_start                    str
terminal_end                      str
turn                              str
document_id                    string
document_amount               float64
tip                           float64
document_total                float64
dtype: object

(1697, 11)

In [7]:
# cargamos la venta total de artículos agregada -> devuelve un dataframe

total_articles = read_article_sales(BRONZE / 'TOTAL_ARTICULOS.xls')

# comprobamos que se han cargado bien
display(total_articles.head(10))
display(total_articles.dtypes)
display(total_articles.shape)

,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,department_code,department_name,article_code,article_name,units,amount
0,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,1901,EMPANADILLA CRIOLLA,1359.0,5425.59990
1,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2031,ROSSINI,634.0,4322.72609
2,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2303,BOCATIN DE CALAMARES,1188.0,7668.00162
3,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2304,MINI TORTILLA CON TRUFA,1.0,7.27273
4,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2306,PAN BAO DE COCHINITA PIBIL,31.0,183.18180
5,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2373,GYOZA DE CERDO,733.0,6016.36303
6,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2378,TIGRE DE BOGAVANTE,945.0,5988.18215
7,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2381,PAN BAO DE CARRILLERA DE CERDO,2041.0,12612.76485
8,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,1,TAPAS CALIENTES,2382,ALBONDIGA XL DE WAGYU,914.0,6638.54556
9,TOTAL_ARTICULOS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2,TAPAS FRIAS,11,BOMBONES DE FOIE,511.0,3158.91000


source_file                       str
report_start           datetime64[us]
report_end             datetime64[us]
report_generated_on    datetime64[us]
terminal_start                    str
terminal_end                      str
turn                              str
department_code                 Int64
department_name                   str
article_code                    Int64
article_name                      str
units                         float64
amount                        float64
dtype: object

(251, 13)

## 2. Carga de catálogo

In [8]:
# cargamos la lista de departamentos -> devuelve un dataframe

df_depart = read_departments(BRONZE / 'DEPARTAMENTOS.xls')

# comprobamos que se han cargado bien
display(df_depart.head(10))
display(df_depart.dtypes)
display(df_depart.shape)

,source_file,department_code,department_name,department_short_name
0,DEPARTAMENTOS.xls,29,APARTADO,<NA>
1,DEPARTAMENTOS.xls,8,BLANCOS,<NA>
2,DEPARTAMENTOS.xls,7,BODEGA,<NA>
3,DEPARTAMENTOS.xls,3,CARTA,<NA>
4,DEPARTAMENTOS.xls,5,CERVEZAS,<NA>
5,DEPARTAMENTOS.xls,28,CLASICOS,<NA>
6,DEPARTAMENTOS.xls,10,ESPUMOSOS,<NA>
7,DEPARTAMENTOS.xls,22,EXTRAS,<NA>
8,DEPARTAMENTOS.xls,12,FIESTAS,<NA>
9,DEPARTAMENTOS.xls,26,GUARNICIONES,<NA>


source_file                 str
department_code           Int64
department_name          string
department_short_name    string
dtype: object

(23, 4)

In [9]:
# cargamos la lista de artículos -> devuelve un dataframe

df_artic = read_articles(BRONZE / 'ARTICULOS.xls')

# comprobamos que se han cargado bien
display(df_artic.head(10))
display(df_artic.dtypes)
display(df_artic.shape)

,source_file,article_code,article_name,article_short_name,department_code
0,ARTICULOS.xls,2195,MEDIA DE BERBERECHOS A LA BRASA,<NA>,22
1,ARTICULOS.xls,1662,TORTILLA DE BETANZOS,<NA>,14
2,ARTICULOS.xls,2256,200 MONJES RESERVA,<NA>,29
3,ARTICULOS.xls,2364,A TRABA GODELLO,<NA>,8
4,ARTICULOS.xls,2266,ABADIA DEL POBLET,<NA>,29
5,ARTICULOS.xls,2259,ABADIA RETUERTA,<NA>,9
6,ARTICULOS.xls,2156,ABUELO 7,<NA>,6
7,ARTICULOS.xls,2269,ACOROA LIAS,<NA>,29
8,ARTICULOS.xls,2388,AGUA RECICLABLE,<NA>,4
9,ARTICULOS.xls,2309,AGUA 70CL,<NA>,4


source_file              str
article_code           Int64
article_name          string
article_short_name    string
department_code        Int64
dtype: object

(405, 5)

In [10]:
# cargamos el menú/carta -> devuelve un dataframe

df_menu = read_menu(BRONZE / 'CARTA.xls')

# comprobamos que se han cargado bien
display(df_menu.head(10))
display(df_menu.dtypes)
display(df_menu.shape)

,source_file,article_code,article_name
0,CARTA.xls,2195,MEDIA DE BERBERECHOS A LA BRASA
1,CARTA.xls,1662,TORTILLA DE BETANZOS
2,CARTA.xls,2256,200 MONJES RESERVA
3,CARTA.xls,2364,A TRABA GODELLO
4,CARTA.xls,2266,ABADIA DEL POBLET
5,CARTA.xls,2259,ABADIA RETUERTA
6,CARTA.xls,2156,ABUELO 7
7,CARTA.xls,2269,ACOROA LIAS
8,CARTA.xls,2388,AGUA RECICLABLE
9,CARTA.xls,2309,AGUA 70CL


source_file        str
article_code     Int64
article_name    string
dtype: object

(405, 3)

## 3. Carga de fuentes externas

In [11]:
# El CSV tiene dos secciones separadas por una línea en blanco:
#   - Bloque horario:  temperatura, código meteorológico y lluvia por hora
#   - Bloque diario:   resumen diario con máx, mín, media, precipitación, viento y sol
#
# Como la predicción es a granularidad diaria, el bloque diario es la fuente
# principal para el modelo. El bloque horario se conserva en Bronze por si
# fuera necesario en análisis posteriores

RUTA_METEO = BRONZE / "open-meteo-pozuelo.csv"

# --- Localizar el inicio de cada bloque ---
with open(RUTA_METEO, encoding="utf-8") as f:
    lineas = f.readlines()

indices_time = [i for i, linea in enumerate(lineas) if linea.startswith("time")]

if len(indices_time) != 2:
    raise ValueError(
        f"Se esperaban 2 bloques en el CSV meteorológico, "
        f"se encontraron {len(indices_time)}. Revisar el fichero."
    )

idx_horario = indices_time[0]   # base-0: línea 3
idx_diario  = indices_time[1]   # base-0: línea 22109

# --- Bloque horario ---
# nrows = filas entre cabecera horaria y línea en blanco antes del bloque diario
nrows_horario = idx_diario - idx_horario - 2  # -1 cabecera, -1 línea en blanco

meteo_horaria = pd.read_csv(
    RUTA_METEO,
    skiprows=idx_horario,
    nrows=nrows_horario,
)

meteo_horaria = meteo_horaria.rename(columns={"time": "datetime"})
meteo_horaria["datetime"] = pd.to_datetime(meteo_horaria["datetime"])

# --- Bloque diario ---
meteo_diaria = pd.read_csv(
    RUTA_METEO,
    skiprows=idx_diario,
)

meteo_diaria = meteo_diaria.rename(columns={"time": "date"})
meteo_diaria["date"] = pd.to_datetime(meteo_diaria["date"]).dt.date

# --- Verificación ---
print(f"Horaria — filas: {len(meteo_horaria):,}  |  "
      f"desde: {meteo_horaria['datetime'].min()}  "
      f"hasta: {meteo_horaria['datetime'].max()}")

print(f"Diaria  — filas: {len(meteo_diaria):,}  |  "
      f"desde: {meteo_diaria['date'].min()}  "
      f"hasta: {meteo_diaria['date'].max()}")

display(meteo_horaria.head(3))
display(meteo_diaria.head(3))

Horaria — filas: 30,864  |  desde: 2023-01-01 00:00:00  hasta: 2026-07-09 23:00:00
Diaria  — filas: 1,286  |  desde: 2023-01-01  hasta: 2026-07-09


,datetime,temperature_2m (°C),weather_code (wmo code),rain (mm)
0,2023-01-01 00:00:00,4.2,0,0.0
1,2023-01-01 01:00:00,3.8,2,0.0
2,2023-01-01 02:00:00,4.2,1,0.0


,date,temperature_2m_mean (°C),temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),rain_sum (mm),precipitation_hours (h),wind_speed_10m_max (km/h),sunshine_duration (s)
0,2023-01-01,7.7,13.7,2.9,0.0,0.0,0.0,11.1,28033.79
1,2023-01-02,8.9,11.9,6.8,2.2,2.2,9.0,9.7,20630.37
2,2023-01-03,6.4,11.7,2.6,0.0,0.0,0.0,7.9,32135.08


In [12]:
# cargamos los festivos oficiales en madrid 

df_festivos = pd.read_csv(BRONZE/"festivos_madrid.csv")

# comprobamos que se han cargado bien
display(df_festivos.head(10))
display(df_festivos.dtypes)
display(df_festivos.shape)

,fecha,festivo_nombre,es_festivo,dia_semana,nivel
0,2023-01-01,Año Nuevo,1,Domingo,nacional_autonomico
1,2023-01-06,Epifanía del Señor,1,Viernes,nacional_autonomico
2,2023-03-20,Lunes siguiente a San José,1,Lunes,nacional_autonomico
3,2023-04-06,Jueves Santo,1,Jueves,nacional_autonomico
4,2023-04-07,Viernes Santo,1,Viernes,nacional_autonomico
5,2023-05-01,Fiesta del Trabajo,1,Lunes,nacional_autonomico
6,2023-05-02,Fiesta de la Comunidad de Madrid,1,Martes,nacional_autonomico
7,2023-05-15,San Isidro (festivo local Madrid capital),1,Lunes,local_madrid_capital
8,2023-07-17,Virgen del Carmen (festivo local Pozuelo),1,Lunes,local_pozuelo
9,2023-08-15,Asunción de la Virgen,1,Martes,nacional_autonomico


fecha               str
festivo_nombre      str
es_festivo        int64
dia_semana          str
nivel               str
dtype: object

(57, 5)

In [13]:
# cargamos los eventos relevantes en pozuelo durante el periodo observado
# aquí tenemos puentes, días señalados, vacaciones, evento culturales y deportivos, etc

df_eventos = pd.read_csv(BRONZE/"eventos_relevantes_pozuelo_la_roca_2025_2026.csv")

# comprobamos que se han cargado bien
display(df_eventos.head(10))
display(df_eventos.dtypes)
display(df_eventos.shape)

,event_id,fecha_inicio,fecha_fin,nombre_evento,categoria,subcategoria,ambito,ubicacion,proximidad_la_roca,impacto_esperado,intensidad_sugerida,direccion_demanda,franja_probable,segmento_cliente,confianza_fecha,estimado,feature_sugerida,fuente_tipo,source_url,notas_modelado
0,EVT_0001,2025-10-10,2025-10-13,Puente escolar de octubre / Hispanidad en domingo,calendario_escolar,puente_no_lectivo,Comunidad de Madrid,Pozuelo / Comunidad de Madrid,alta,medio,2,mixto,"comida,cena","familias,residentes",alta,0,puente_octubre_escolar,oficial,https://www.comunidad.madrid/educacion/calenda...,El 13/10/2025 figura como dia no lectivo en el...
1,EVT_0002,2025-10-19,2025-10-19,12a Carrera Popular Ciudad de Pozuelo,evento_deportivo_local,running,Pozuelo,Polideportivo El Torreon y calles principales,media,medio,2,aumenta,"manana,comida","familias,deportistas,acompanantes",alta,0,evento_running_local,oficial,https://www.pozuelodealarcon.org/agenda/12a-ca...,Evento local de domingo; posible incremento en...
2,EVT_0003,2025-10-21,2025-10-22,Champions League 2025/26 - jornada 3,evento_deportivo_tv,futbol_champions,Europa / TV,Visionado en restaurantes,baja,medio,2,aumenta,"cena,noche","aficionados,grupos",alta,0,champions_league,oficial,https://www.uefa.com/uefachampionsleague/news/...,Fechas oficiales UEFA; impacto operativo depen...
3,EVT_0004,2025-10-24,2025-10-26,Fin de semana de El Clasico en Madrid,evento_deportivo_tv,futbol_laliga,Madrid / TV,Santiago Bernabeu y visionado en restaurantes,baja,medio,2,aumenta,"tarde,cena","aficionados,grupos",alta,0,futbol_clasico_madrid,oficial,https://www.realmadrid.com/es-ES/noticias/futb...,El partido Real Madrid-Barcelona fue el 26/10/...
4,EVT_0005,2025-10-31,2025-10-31,Halloween,calendario_social,fecha_senalada,General,Pozuelo / Madrid,media,medio,2,aumenta,"tarde,cena,noche","familias,jovenes,grupos",alta,0,halloween,regla_calendario,regla_calendario_estandar,"Fecha no festiva con efecto en ocio familiar, ..."
5,EVT_0006,2025-10-31,2025-11-03,Puente de Todos los Santos / no lectivo escolar,calendario_escolar,puente_no_lectivo,Comunidad de Madrid,Pozuelo / Comunidad de Madrid,alta,medio,2,mixto,"comida,cena","familias,residentes",alta,0,puente_todos_los_santos,oficial,https://www.comunidad.madrid/educacion/calenda...,El 03/11/2025 figura como dia no lectivo; pued...
6,EVT_0007,2025-11-04,2025-11-05,Champions League 2025/26 - jornada 4,evento_deportivo_tv,futbol_champions,Europa / TV,Visionado en restaurantes,baja,medio,2,aumenta,"cena,noche","aficionados,grupos",alta,0,champions_league,oficial,https://www.uefa.com/uefachampionsleague/news/...,Fechas oficiales UEFA; impacto operativo depen...
7,EVT_0008,2025-11-25,2025-11-26,Champions League 2025/26 - jornada 5,evento_deportivo_tv,futbol_champions,Europa / TV,Visionado en restaurantes,baja,medio,2,aumenta,"cena,noche","aficionados,grupos",alta,0,champions_league,oficial,https://www.uefa.com/uefachampionsleague/news/...,Fechas oficiales UEFA; impacto operativo depen...
8,EVT_0009,2025-11-28,2025-11-30,Black Friday y fin de semana comercial,calendario_comercial,compras,General / Madrid,Zonas comerciales y restauracion,media,bajo,1,mixto,"comida,tarde,cena","compradores,familias",alta,0,black_friday,regla_modelado,regla_modelado_negocio,Evento comercial; util para capturar posible i...
9,EVT_0010,2025-12-01,2025-12-19,Campana de cenas y comidas de empresa de Navidad,temporada_modelado,navidad_empresa,General / Madrid,Restauracion,alta,alto,3,aumenta,"comida,cena","empresas,grupos",media,1,cenas_empresa_navidad,regla_modelado,regla_modelado_negocio,Rango heuristico para comidas y cenas de empre...


event_id                 str
fecha_inicio             str
fecha_fin                str
nombre_evento            str
categoria                str
subcategoria             str
ambito                   str
ubicacion                str
proximidad_la_roca       str
impacto_esperado         str
intensidad_sugerida    int64
direccion_demanda        str
franja_probable          str
segmento_cliente         str
confianza_fecha          str
estimado               int64
feature_sugerida         str
fuente_tipo              str
source_url               str
notas_modelado           str
dtype: object

(78, 20)

## 4. Snapshots a Parquet (BRONZE)

Guardamos los dataframes sin modificar (raw) en formato parquet para que la depuración en silver sea más ágil 

Silver no tiene que volver a leer los xls, xlsx, csv

Algunas columnas tienen tipos mezclados (strings y objects), pero parquet no acepta eso y da error. 

El problema concreto: la columna `entered_by` en `df_reservas` tiene tipo `object` en pandas pero contiene valores mixtos: algunos son strings y otros son enteros. Parquet no acepta columnas mixtas.

Esto es exactamente el tipo de problema que se resuelve en Silver, no en Bronze. Pero el snapshot de Bronze también tiene que guardarse, así que la solución aquí es forzar todo a string antes de guardar, sin perder información:

In [14]:
def preparar_para_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte columnas object con tipos mixtos a string puro.
    Solo afecta al snapshot — no modifica el DataFrame original.
    """
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

In [15]:
# Guardamos cada DataFrame como Parquet en data/bronze/snapshots/.
# A partir de aquí, Silver y Gold cargan desde Parquet — nunca desde XLS o CSV.
# El sufijo _raw indica que son datos crudos sin depurar.

BRONZE_SNAPSHOTS = BRONZE / "snapshots"
BRONZE_SNAPSHOTS.mkdir(parents=True, exist_ok=True)

snapshots = {
    "tickets_raw":       df_tickets,
    "ventas_raw":        df_ventas,
    "reservas_raw":      df_reservas,
    "tips_raw":          df_tips,
    "articulos_raw":     df_artic,
    "departamentos_raw": df_depart,
    "menu_raw":          df_menu,
    "festivos_raw":      df_festivos,
    "eventos_raw":       df_eventos,
    "meteo_horaria_raw": meteo_horaria,
    "meteo_diaria_raw":  meteo_diaria,
    "total_articles_raw": total_articles,
}

for nombre, df in snapshots.items():
    ruta = BRONZE_SNAPSHOTS / f"{nombre}.parquet"
    preparar_para_parquet(df).to_parquet(ruta, index=False)
    print(f"✓  {nombre}.parquet  —  {len(df):,} filas  x  {len(df.columns)} columnas")

✓  tickets_raw.parquet  —  6,709 filas  x  11 columnas
✓  ventas_raw.parquet  —  5,532 filas  x  13 columnas
✓  reservas_raw.parquet  —  21,341 filas  x  20 columnas
✓  tips_raw.parquet  —  1,697 filas  x  11 columnas
✓  articulos_raw.parquet  —  405 filas  x  5 columnas
✓  departamentos_raw.parquet  —  23 filas  x  4 columnas
✓  menu_raw.parquet  —  405 filas  x  3 columnas
✓  festivos_raw.parquet  —  57 filas  x  5 columnas
✓  eventos_raw.parquet  —  78 filas  x  20 columnas
✓  meteo_horaria_raw.parquet  —  30,864 filas  x  4 columnas
✓  meteo_diaria_raw.parquet  —  1,286 filas  x  9 columnas
✓  total_articles_raw.parquet  —  251 filas  x  13 columnas


/var/folders/8p/w0tfqpks0xn1c0tbzdbrcdg00000gr/T/ipykernel_31553/1189800153.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
/var/folders/8p/w0tfqpks0xn1c0tbzdbrcdg00000gr/T/ipykernel_31553/1189800153.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pand

## 5. Siguiente paso

Los snapshots están disponibles en `data/bronze/snapshots/`.
Continuar en `02_depuracion_silver.ipynb`.

TODO: tfm_pdf_agent.py — pendiente de implementar.

Los PDFs en data/bronze/pdfs/ contienen tickets individuales.

Se usarán para validar y enriquecer df_tickets, no como fuente primaria.

LISTA_TICKETS.xls es la fuente principal de datos de tickets.